In [3]:
from piq import psnr, ssim

In [4]:
import os
import random
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from PIL import Image
import matplotlib.pyplot as plt
import cv2

# 🔧 PyTorch and Metrics
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import vgg19
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.utils import save_image
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [5]:
from tqdm import tqdm

In [6]:
# ⚙️ Global Configuration
class CFG:
    # Data Paths
    ROOT_DIR = "/kaggle/input/manga109s/Manga109s"
    TRAIN_CSV = os.path.join(ROOT_DIR, "train.csv")
    VAL_CSV = os.path.join(ROOT_DIR, "val.csv")
    TEST_CSV = os.path.join(ROOT_DIR, "test.csv")
    
    # Training Hyperparameters
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    EPOCHS = 30 # A higher resolution and GAN needs more time to converge
    
    BATCH_SIZE = 1 
    GRAD_ACCUM_STEPS = 4 # Effective batch size of 4

    # <<< --- THE UPGRADE --- >>>
    IMAGE_SIZE = 256 # Target resolution
    
    NUM_WORKERS = 2
    LR = 2e-4
    SEED = 43
    EPOCH_SWITCH = 15

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(CFG.SEED)
print(f"✅ Setup complete for {CFG.IMAGE_SIZE}x{CFG.IMAGE_SIZE} training. Using device: {CFG.DEVICE}")

✅ Setup complete for 256x256 training. Using device: cuda


In [7]:
def parse_text_bboxes(annotation_path, page_index, image_size, original_size):
    """Extracts and scales text bounding boxes from Manga109 XML annotation."""
    try:
        tree = ET.parse(annotation_path)
        root = tree.getroot()
        pages = root.findall('page')
        if page_index >= len(pages):
            return []
            
        original_w, original_h = original_size
        scale_x = image_size / original_w
        scale_y = image_size / original_h
        
        boxes = []
        for text in pages[page_index].findall('text'):
            x = int(text.get("x"))
            y = int(text.get("y"))
            w = int(text.get("width"))
            h = int(text.get("height"))
            # Scale the box coordinates
            boxes.append((int(x * scale_x), int(y * scale_y), int((x + w) * scale_x), int((y + h) * scale_y)))
        return boxes
    except (FileNotFoundError, ET.ParseError):
        return []

def generate_irregular_mask_excluding_boxes(shape, exclusion_boxes, **kwargs):
    """Generate random irregular mask, ensuring exclusion zones are not masked."""
    height, width = shape
    mask = np.zeros((height, width), dtype=np.uint8)
    for _ in range(random.randint(1, 5)):
        num_vertex = random.randint(5, 12)
        start_x = random.randint(0, width)
        start_y = random.randint(0, height)
        brush_width = random.randint(10, 20)
        angle = random.uniform(0, 2 * np.pi)
        for _ in range(num_vertex):
            angle += random.uniform(-np.pi/3, np.pi/3)
            length = random.randint(20, 70)
            end_x = np.clip(int(start_x + length * np.cos(angle)), 0, width - 1)
            end_y = np.clip(int(start_y + length * np.sin(angle)), 0, height - 1)
            cv2.line(mask, (start_x, start_y), (end_x, end_y), 1, brush_width)
            start_x, start_y = end_x, end_y
            
    for (x1, y1, x2, y2) in exclusion_boxes:
        mask[y1:y2, x1:x2] = 0 # Set exclusion zones to 0
    return torch.from_numpy(mask).float().unsqueeze(0)

class Manga109InpaintingDataset(Dataset):
    def __init__(self, csv_path, root_dir, image_size):
        self.data = pd.read_csv(csv_path)
        self.root_dir = root_dir
        self.image_size = image_size
        
        # CRITICAL FIX 1: Add Normalize for [-1, 1] range.
        self.transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),
            T.Normalize(mean=[0.5], std=[0.5])
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image_path = os.path.join(self.root_dir, row['image_path'])
        annotation_path = os.path.join(self.root_dir, row.get('annotation_path', ''))
        page_index = int(row.get('page_index', 0))

        image = Image.open(image_path).convert('L')
        original_size = image.size
        image_tensor = self.transform(image)
        
        exclusion_boxes = parse_text_bboxes(annotation_path, page_index, self.image_size, original_size)
        mask_tensor = generate_irregular_mask_excluding_boxes((self.image_size, self.image_size), exclusion_boxes)

        # CRITICAL FIX 2: Use zero-fill masking instead of white-fill.
        masked_image = image_tensor * (1.0 - mask_tensor)

        return {
            'image': image_tensor,
            'masked_image': masked_image,
            'mask': mask_tensor,
        }




In [8]:
# This is the final, memory-safe architecture for 256x256 training.
# It uses Gradient Checkpointing to trade speed for a massive reduction in memory usage.

from torch.utils.checkpoint import checkpoint

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class Attention(nn.Module):
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        x = F.scaled_dot_product_attention(q, k, v)
        x = x.transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x

class GatedFFN(nn.Module):
    def __init__(self, dim, mlp_ratio=4.):
        super().__init__()
        hidden_features = int(dim * mlp_ratio)
        self.fc1 = nn.Linear(dim, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, dim)
    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = GatedFFN(dim)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class TFormerUNet(nn.Module):
    # Pruned to be "thinner" as a safety margin.
    def __init__(self, in_ch=1, base_dim=24):
        super().__init__()
        
        # --- Encoder (Deeper for 256x256) ---
        self.enc1 = ConvBlock(in_ch, base_dim)               # -> 24, 256, 256
        self.down1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base_dim, base_dim * 2)        # -> 48, 128, 128
        self.down2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base_dim * 2, base_dim * 4)    # -> 96, 64, 64
        self.down3 = nn.MaxPool2d(2)
        
        # --- Bottleneck (Transformer) ---
        self.proj_in = nn.Conv2d(base_dim * 4, base_dim * 8, 1) # -> 192, 32, 32
        self.transformer = TransformerBlock(base_dim * 8, num_heads=8)
        self.proj_out = nn.Conv2d(base_dim * 8, base_dim * 4, 1) # -> 96, 32, 32

        # --- Decoder (Deeper) ---
        self.up3 = nn.ConvTranspose2d(base_dim * 4, base_dim * 2, 2, stride=2)
        self.dec3 = ConvBlock(base_dim * 6, base_dim * 2) # Skip from enc3 (96) + up3 (48) = 144
        self.up2 = nn.ConvTranspose2d(base_dim * 2, base_dim, 2, stride=2)
        self.dec2 = ConvBlock(base_dim * 3, base_dim)      # Skip from enc2 (48) + up2 (24) = 72
        self.up1 = nn.ConvTranspose2d(base_dim, base_dim, 2, stride=2)
        self.dec1 = ConvBlock(base_dim * 2, base_dim)      # Skip from enc1 (24) + up1 (24) = 48
        
        self.final_conv = nn.Conv2d(base_dim, in_ch, 1)

    def forward(self, x):
        # --- Encoder with Gradient Checkpointing ---
        # We apply checkpointing to the most memory-intensive blocks.
        skip1 = checkpoint(self.enc1, x, use_reentrant=False)
        down1 = self.down1(skip1)
        skip2 = checkpoint(self.enc2, down1, use_reentrant=False)
        down2 = self.down2(skip2)
        skip3 = checkpoint(self.enc3, down2, use_reentrant=False)
        down3 = self.down3(skip3)
        
        # --- Bottleneck ---
        B, C, H, W = down3.shape
        bn = self.proj_in(down3)
        bn_seq = bn.flatten(2).transpose(1, 2)
        # Checkpoint the transformer as well
        bn_seq = checkpoint(self.transformer, bn_seq, use_reentrant=False)
        bn_img = bn_seq.transpose(1, 2).view(B, -1, H, W)
        bn = self.proj_out(bn_img)

        # --- Decoder ---
        up3 = self.up3(bn)
        dec3 = self.dec3(torch.cat([up3, skip3], 1))
        up2 = self.up2(dec3)
        dec2 = self.dec2(torch.cat([up2, skip2], 1))
        up1 = self.up1(dec2)
        dec1 = self.dec1(torch.cat([up1, skip1], 1))
        
        return torch.tanh(self.final_conv(dec1))

In [9]:
# --- Cell 5b: Discriminator with Feature Extraction Capability ---
class Discriminator(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        self.block1 = self._discriminator_block(in_channels, 64, norm=False)
        self.block2 = self._discriminator_block(64, 128)
        self.block3 = self._discriminator_block(128, 256)
        self.block4 = self._discriminator_block(256, 512, stride=1)
        self.final_conv = nn.Conv2d(512, 1, 4, padding=1)
    def _discriminator_block(self, in_filters, out_filters, stride=2, norm=True):
        layers = [nn.Conv2d(in_filters, out_filters, 4, stride=stride, padding=1)]
        if norm: layers.append(nn.InstanceNorm2d(out_filters))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        return nn.Sequential(*layers)
    def forward(self, img, return_features=False):
        feat1 = self.block1(img)
        feat2 = self.block2(feat1)
        feat3 = self.block3(feat2)
        feat4 = self.block4(feat3)
        prediction = self.final_conv(feat4)
        if return_features:
            return prediction, [feat1, feat2, feat3, feat4]
        return prediction

In [10]:
# --- Instantiate DataLoaders ---
test_dataset = Manga109InpaintingDataset(CFG.TEST_CSV, CFG.ROOT_DIR, CFG.IMAGE_SIZE)
test_loader = DataLoader(test_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS)

model = TFormerUNet(in_ch=1, base_dim=24).to(CFG.DEVICE)

In [11]:
from prettytable import PrettyTable

def count_parameters(model):
    table = PrettyTable(["Modules", "Parameters"])
    total_params = 0
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        params = parameter.numel()
        table.add_row([name, params])
        total_params += params
    print(table)
    print(f"Total Trainable Params: {total_params}")
    return total_params
    
count_parameters(model)

+------------------------------+------------+
|           Modules            | Parameters |
+------------------------------+------------+
|      enc1.conv.0.weight      |    216     |
|      enc1.conv.1.weight      |     24     |
|       enc1.conv.1.bias       |     24     |
|      enc1.conv.3.weight      |    5184    |
|      enc1.conv.4.weight      |     24     |
|       enc1.conv.4.bias       |     24     |
|      enc2.conv.0.weight      |   10368    |
|      enc2.conv.1.weight      |     48     |
|       enc2.conv.1.bias       |     48     |
|      enc2.conv.3.weight      |   20736    |
|      enc2.conv.4.weight      |     48     |
|       enc2.conv.4.bias       |     48     |
|      enc3.conv.0.weight      |   41472    |
|      enc3.conv.1.weight      |     96     |
|       enc3.conv.1.bias       |     96     |
|      enc3.conv.3.weight      |   82944    |
|      enc3.conv.4.weight      |     96     |
|       enc3.conv.4.bias       |     96     |
|        proj_in.weight        |  

788689

In [14]:
model = TFormerUNet(in_ch=1, base_dim=24).to(CFG.DEVICE)
model.load_state_dict(torch.load('/kaggle/input/inpainting_model/pytorch/v2/1/tformer_manga_best_model_256.pth', map_location=CFG.DEVICE))
model.to(CFG.DEVICE)
model.eval()

TFormerUNet(
  (enc1): ConvBlock(
    (conv): Sequential(
      (0): Conv2d(1, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (down1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (enc2): ConvBlock(
    (conv): Sequential(
      (0): Conv2d(24, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stat

In [15]:
psnr_list = []
ssim_list = []
with torch.no_grad():
    for idx, batch in enumerate(tqdm(test_loader)):
        masked = batch['masked_image'].to(CFG.DEVICE)
        mask = batch['mask'].to(CFG.DEVICE)
        image = batch['image'].to(CFG.DEVICE)

        output = model(masked)
        comp = output * (1 - mask) + masked * mask  # final composite output
        psnr_list.append(psnr(((output + 1) / 2).float(), ((image + 1) / 2).float(), data_range=1.0).item())
        ssim_list.append(ssim(((output + 1) / 2).float(), ((image + 1) / 2).float(), data_range=1.0).item())
        if idx%50==0:
            comparison_grid = torch.cat([
                    (masked.cpu() + 1) / 2,
                    (output.cpu() + 1) / 2,
                    (image.cpu() + 1) / 2
                ])
            save_image(comparison_grid, f"id_{idx}.png", nrow=len(image))

print(f"Average PSNR: {sum(psnr_list)/len(psnr_list):.2f}")
print(f"Average SSIM: {sum(ssim_list)/len(ssim_list):.4f}")


100%|██████████| 907/907 [00:31<00:00, 28.91it/s]

Average PSNR: 24.81
Average SSIM: 0.9312
